## Gaussian Splatting Demo

### Introduction

Traditional robotics mapping produces sparse point clouds (ORB-SLAM) or 2D occupancy grids (SLAM Toolbox). Neither captures visual appearance. A robot with an occupancy grid of a kitchen knows where the walls are but cannot answer *"what does the kitchen look like from the doorway?"* or *"is this the same kitchen I saw before?"*

[3D Gaussian Splatting](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) (3DGS) produces a dense, renderable 3D representation by fitting a collection of 3D Gaussians to a set of posed images. Each Gaussian has a position, covariance (shape), color (via spherical harmonics), and opacity. Rendering is done by splatting these Gaussians onto an image plane — no ray marching needed — enabling real-time (100+ FPS) novel view synthesis.

For robotics, this enables:

- **Visual re-localization** — render what the robot expects to see from a candidate pose, compare with what it actually sees
- **Scene understanding** — with language-embedded splats, the 3D map becomes queryable: *"where is the couch?"* returns a rendered view and a pose
- **Navigation planning** — dense 3D geometry from splats generates richer obstacle representations than sparse point clouds

In this project you will use [gsplat](https://github.com/nerfstudio-project/gsplat), the actively maintained successor library from the Nerfstudio team. gsplat is a high-performance, modular library for differentiable Gaussian splatting — it provides the core rasterization engine, training strategies, and a `simple_trainer` example that serves as a complete training pipeline. Compared to the legacy Nerfstudio wrapper, gsplat is lighter, faster (uses ~4x less memory), and gives you direct access to the underlying primitives.

**An important note on simulation vs. reality.** In Part 2 of this project you will train a Gaussian Splat from images captured by a simulated D435i camera in the Gazebo house world. 3DGS faithfully reconstructs whatever it sees — if the input images are synthetic, the output splat will look synthetic too. A Gaussian Splat cannot make Gazebo renders more photorealistic than they already are. What 3DGS *does* give you, even from synthetic input, is a dense, view-consistent 3D representation that supports novel view synthesis, depth rendering, and semantic queries — capabilities that occupancy grids and sparse point clouds lack.

This works because 3DGS requires **multi-view consistency and accurate camera poses**, not photorealism. Gazebo satisfies both: its deterministic renderer produces perfectly consistent images across viewpoints, and simulated odometry provides ground-truth poses with zero drift. The original 3DGS paper evaluated on the synthetic Blender/NeRF-Synthetic dataset, and all major GS-SLAM systems ([SplaTAM](https://spla-tam.github.io/), [GS-SLAM](https://gs-slam.github.io/), [RTG-SLAM](https://arxiv.org/abs/2404.19706)) benchmark on [Replica](https://wijmans.xyz/publication/replica-2019/) — a synthetic indoor dataset rendered with OpenGL. That said, Gazebo's default rendering quality (Ogre2) is below Replica's level of fidelity, so expect the reconstruction to reflect the visual quality of Gazebo's house world, not a photorealistic scan. The sim-to-real comparison in Task 2.4 is designed to surface exactly this gap.

Gazebo's rendering pipeline does support higher-fidelity materials ([PBR with normal/roughness/metalness maps](https://gazebosim.org/api/rendering/10/classgz_1_1rendering_1_1Ogre2Material.html), [baked light maps with global illumination](https://gazebosim.org/api/rendering/9/lightmap.html), and [configurable sensor noise](https://gazebosim.org/libs/sensors/)), but the default house world does not fully exploit these capabilities.

This demo downloads the **garden** scene from the [Mip-NeRF 360](https://huggingface.co/datasets/mileleap/mipnerf360) benchmark dataset, trains a Gaussian splat model using `simple_trainer.py`, and renders a fly-through video — all in a few minutes on a single GPU.

### Prerequisites

- Completed [Camera Calibration](/aiml-common/lectures/sensor-models/cameras/camera-calibration) lecture
- A camera for scene capture — **any** of the following works:
  - iPhone 12 Pro or newer (LiDAR) with [Record3D](https://record3d.app/) — best quality, no COLMAP needed
  - Any iPhone/Android with [Polycam](https://poly.cam/) or [Scaniverse](https://scaniverse.com/)
  - Laptop webcam or USB camera — use COLMAP for pose estimation (see below)
- Workstation with NVIDIA GPU (RTX 3060+ for preview quality, RTX 3090/4090 for full quality)
- Docker with NVIDIA Container Toolkit installed

### COLMAP Installation

[COLMAP](https://colmap.github.io/) is a Structure-from-Motion (SfM) and Multi-View Stereo (MVS) pipeline that estimates camera poses from unposed images. It is required if you are using a monocular camera (laptop webcam, USB camera, or phone without LiDAR).

Unlike the legacy Nerfstudio workflow (which wrapped COLMAP behind `ns-process-data`), with gsplat you run COLMAP directly. gsplat's dataset parser expects raw COLMAP output (`sparse/0/cameras.bin`, `images.bin`, `points3D.bin`).

**Option A — Inside the gsplat Docker container:**

COLMAP can be installed via `apt` inside the container:

```bash
apt-get update && apt-get install -y colmap
```

Then run the COLMAP pipeline directly:

```bash
# Feature extraction
colmap feature_extractor --database_path db.db --image_path images/
# Feature matching
colmap exhaustive_matcher --database_path db.db
# Sparse reconstruction
mkdir -p sparse/0
colmap mapper --database_path db.db --image_path images/ --output_path sparse/
```

**Option B — Native installation on Ubuntu:**

```bash
sudo apt-get install colmap
```

For GPU-accelerated feature matching (significantly faster on large image sets), build from source with CUDA support. Follow the [official build instructions](https://colmap.github.io/install.html).

**Option C — pip (CPU-only, slower but simplest):**

```bash
pip install pycolmap
```

### Background Reading

Before starting, study these resources in order:

1. **Conceptual overview:** [Introduction to 3D Gaussian Splatting](https://huggingface.co/blog/gaussian-splatting) (Hugging Face blog) — accessible explanation of the theory
2. **Original paper:** [3D Gaussian Splatting for Real-Time Radiance Field Rendering](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) (SIGGRAPH 2023) — the foundational work
3. **Robotics survey:** [3DGS in Robotics: A Survey](https://arxiv.org/abs/2410.12262) — how 3DGS is being used for SLAM, navigation, and manipulation
4. **gsplat documentation and examples:** [github.com/nerfstudio-project/gsplat/tree/main/examples](https://github.com/nerfstudio-project/gsplat/tree/main/examples) — the training framework you will use, including `simple_trainer.py` and dataset parsers

### Compute Requirements

| Task | VRAM | GPU | Time |
|------|------|-----|------|
| Training gsplat, 7K steps (preview) | 4-6 GB | RTX 3060+ | ~5 min |
| Training gsplat, 30K steps (full) | 6-8 GB | RTX 3090/4090 | ~15-20 min |
| Rendering trained splat | — | Any CUDA GPU | 100-200+ FPS |
| COLMAP SfM (200 images) | 2-4 GB | CPU or CUDA GPU | 5-30 min |

---

## Part 0: Working Demo

Before capturing your own data, run this end-to-end demo to verify your environment and see gsplat in action. It downloads the **garden** scene from the [Mip-NeRF 360](https://huggingface.co/datasets/mileleap/mipnerf360) benchmark dataset, trains a Gaussian splat, and renders a fly-through video — all in a few minutes on a single GPU.

This gives you a working baseline to compare against when you train on your own captures in Parts 1 and 2.

The demo runs on **Google Colab** (T4/A100) or your local **Docker GPU** environment.

In [ ]:
# Environment detection
import sys, os, shutil

IN_COLAB = "google.colab" in sys.modules
WORKDIR = "/content" if IN_COLAB else os.getcwd()
DATA_ROOT = os.path.join("/content" if IN_COLAB else os.path.expanduser("~"), "data")
RESULT_DIR = os.path.join(WORKDIR, "results")
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Docker/Local'}")
print(f"Working directory: {WORKDIR}")
print(f"Data root: {DATA_ROOT}")

In [ ]:
# Install dependencies
import sys, subprocess, shutil

IN_COLAB = "google.colab" in sys.modules

def pip_install(*packages):
    """Install packages using uv (if available) or pip."""
    if shutil.which("uv"):
        subprocess.check_call(["uv", "pip", "install", "-q"] + list(packages))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(packages))

if IN_COLAB:
    pip_install("torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121")

# Install gsplat if not already available
try:
    import gsplat
    print(f"gsplat {gsplat.__version__} already installed")
except ImportError:
    print("Installing gsplat...")
    pip_install("gsplat")

import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone gsplat examples and install dependencies
import subprocess, sys, shutil

GSPLAT_DIR = os.path.join(WORKDIR, "gsplat")

if not os.path.exists(GSPLAT_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", "--recurse-submodules", "--branch", "v1.5.3",
        "https://github.com/nerfstudio-project/gsplat.git", GSPLAT_DIR])
else:
    print(f"gsplat repo already cloned at {GSPLAT_DIR}")

def uv_or_pip(*args, no_build_isolation=False):
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "-q"] + list(args)
        if no_build_isolation:
            cmd.insert(4, "--no-build-isolation")
        subprocess.check_call(cmd)
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
        if no_build_isolation:
            cmd.insert(5, "--no-build-isolation")
        subprocess.check_call(cmd)

print("Installing gsplat example dependencies...")

# PyPI packages
uv_or_pip(
    "numpy<2.0.0", "scipy", "scikit-learn", "opencv-python", "Pillow", "piexif",
    "imageio[ffmpeg]", "tensorboard", "torchmetrics[image]", "tensorly",
    "tyro>=0.8.8", "pyyaml", "tqdm", "matplotlib", "splines", "viser",
)

# Git packages that need torch at build time (--no-build-isolation)
uv_or_pip("git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e")
uv_or_pip("git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787")
uv_or_pip("git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5",
           no_build_isolation=True)
uv_or_pip("git+https://github.com/harry7557558/fused-bilagrid@49f0ef06c9f81810fb9b5dd9027cf1844950cc16",
           no_build_isolation=True)

# Verify
trainer_path = os.path.join(GSPLAT_DIR, "examples", "simple_trainer.py")
assert os.path.exists(trainer_path), f"simple_trainer.py not found at {trainer_path}"
print(f"Trainer: {trainer_path}")

import tyro
print(f"tyro {tyro.__version__} OK")

In [ ]:
# Download the garden scene from Mip-NeRF 360 dataset
#
# gsplat expects COLMAP format:
#   <data_dir>/sparse/0/{cameras.bin, images.bin, points3D.bin}
#   <data_dir>/images/      (full-res)
#   <data_dir>/images_4/    (4x downsampled, optional)

from huggingface_hub import snapshot_download

SCENE = "garden"  # other options: bicycle, bonsai, counter, kitchen, room, stump

scene_dir = os.path.join(DATA_ROOT, "360_v2", SCENE)

if not os.path.exists(os.path.join(scene_dir, "sparse")):
    snapshot_download(
        repo_id="mileleap/mipnerf360",
        repo_type="dataset",
        allow_patterns=f"{SCENE}/**",
        local_dir=os.path.join(DATA_ROOT, "360_v2"),
    )
    print(f"Downloaded scene '{SCENE}'")
else:
    print(f"Scene '{SCENE}' already downloaded")

# Verify COLMAP files
sparse_dir = os.path.join(scene_dir, "sparse", "0")
for f in ["cameras.bin", "images.bin", "points3D.bin"]:
    path = os.path.join(sparse_dir, f)
    assert os.path.exists(path), f"Missing {path}"
print(f"COLMAP files verified in {sparse_dir}")

# Show image count
images_dir = os.path.join(scene_dir, "images")
n_images = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.png'))])
print(f"Images: {n_images}")

In [ ]:
# Train the Gaussian splat model
#
# In Docker/headless: train for a limited number of steps.
# In Colab: train for more steps (adjust as needed for your GPU).

MAX_STEPS = 2000 if not IN_COLAB else 7000
DATA_FACTOR = 4  # use 4x downsampled images

result_dir = os.path.join(RESULT_DIR, SCENE)

train_cmd = (
    f"cd {GSPLAT_DIR}/examples && "
    f"python simple_trainer.py default"
    f" --data_dir {scene_dir}"
    f" --data_factor {DATA_FACTOR}"
    f" --result_dir {result_dir}"
    f" --max_steps {MAX_STEPS}"
    f" --save_steps {MAX_STEPS}"
    f" --eval_steps {MAX_STEPS}"
    f" --disable_viewer"
)

print(f"Training for {MAX_STEPS} steps with data_factor={DATA_FACTOR}")
print(f"Command: {train_cmd}")
!{train_cmd}

In [ ]:
# Render a fly-through video from the trained model
import glob

# Find the latest checkpoint
ckpt_pattern = os.path.join(result_dir, "ckpts", "ckpt_*.pt")
ckpts = sorted(glob.glob(ckpt_pattern))

if not ckpts:
    print("No checkpoints found. Complete training first.")
else:
    latest_ckpt = ckpts[-1]
    print(f"Using checkpoint: {latest_ckpt}")

    render_cmd = (
        f"cd {GSPLAT_DIR}/examples && "
        f"python simple_trainer.py default"
        f" --data_dir {scene_dir}"
        f" --data_factor {DATA_FACTOR}"
        f" --result_dir {result_dir}"
        f" --ckpt {latest_ckpt}"
        f" --render_traj_path ellipse"
        f" --disable_viewer"
    )

    print(f"Rendering ellipse trajectory...")
    !{render_cmd}

    # Check for rendered video
    videos = glob.glob(os.path.join(result_dir, "**", "*.mp4"), recursive=True)
    if videos:
        print(f"\nRendered video: {videos[0]}")
    else:
        print("No video output found — check render logs above.")

In [ ]:
# Display the rendered video (if available)
from IPython.display import Video, display

videos = glob.glob(os.path.join(result_dir, "**", "*.mp4"), recursive=True)
if videos:
    display(Video(videos[0], embed=True, width=640))
else:
    print("No rendered video to display.")